In [ ]:
#Google Colab setup for unsloth

%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#Lightning.ai setup for unsloth

%pip uninstall -y unsloth unsloth_zoo trl transformers
%pip install --no-cache-dir "transformers==4.56.2" "trl==0.22.2"
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
# For Windows locally
import os, sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DATASETS_MULTITHREADING_MAX_WORKERS"] = "1"

if sys.platform == "win32":
    from datasets import Dataset

    if not hasattr(Dataset, "_original_map_windows_fix"):
        Dataset._original_map_windows_fix = Dataset.map

        def _win32_safe_map(self, *args, **kwargs):
            kwargs["num_proc"] = None   # force single-process map
            return Dataset._original_map_windows_fix(self, *args, **kwargs)

        Dataset.map = _win32_safe_map

# Minesweeper 5×5 → 6×6 Curriculum Training

This notebook performs a sequential two-size curriculum:

1. Clean 5×5 thinking data.
2. Supervised fine-tuning for two epochs.
3. Save the SFT LoRA adapter.
4. Continue with GRPO on reproducible live 5×5 game states.
5. Continue the trained 5×5 GRPO policy with GRPO on live 6×6 game states.


In [ ]:
from pathlib import Path
import os
import torch
from unsloth import FastLanguageModel

SEED = 42
MODEL_ID = "Qwen/Qwen3-0.6B"
MAX_SEQ_LENGTH = 8192
LORA_RANK = 32

RUN_SFT = True
RUN_5X5_GRPO = True
RUN_6X6_GRPO = True
RELOAD_SAVED_SFT_FOR_5X5_GRPO = False
RELOAD_SAVED_5X5_GRPO_FOR_6X6 = False
RUN_STAGE_EVALUATION = False

SFT_EPOCHS = 2
FIVE_BY_FIVE_GRPO_NUM_EXAMPLES = 500
FIVE_BY_FIVE_GRPO_MAX_STEPS = 500
SIX_BY_SIX_GRPO_NUM_EXAMPLES = 500
SIX_BY_SIX_GRPO_MAX_STEPS = 500
GRPO_MAX_COMPLETION_LENGTH = 512

REPORT_TO = "wandb" if os.getenv("WANDB_API_KEY") else "none"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.75,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.generation_config.max_length = None
print(f"Loaded {MODEL_ID} in 4-bit with LoRA rank {LORA_RANK} and context {MAX_SEQ_LENGTH}.")


## Prompt and chat template


In [ ]:
THINKING_SYSTEM_PROMPT = """You are playing Minesweeper.

Rules:
- Board coordinates are 0-indexed.
- In JSON, x is the column and y is the row, matching board[y][x].
- . and _ mean unknown/hidden.
- F means flagged.
- Numbers 0-8 are revealed cells.
- Only choose hidden cells.
- Choose exactly one action.

Allowed actions:
{"action":"reveal","x":int,"y":int}
{"action":"flag","x":int,"y":int}

Reasoning protocol:
1. Read only the current Board State supplied by the user.
2. Work through the visible numbered constraints before choosing.
3. Put the reasoning inside <think>...</think>.
4. After </think>, output exactly one legal JSON move and no other text.
5. Prefer a forced flag, then a forced safe reveal; guess only if no deterministic move exists.
"""

def get_chat_template():
    return """
    {%- set eos = eos_token if eos_token is defined and eos_token is string and eos_token != '<|im_end|>' else '' -%}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}
    {%- endif %}
    {%- for message in messages %}
        {%- if message.content is string %}
            {%- set content = message.content %}
        {%- else %}
            {%- set content = '' %}
        {%- endif %}
        {%- if message.role == "user" %}
            {{- '<|im_start|>user\\n' + content + '<|im_end|>\\n' }}
        {%- elif message.role == "system" and not loop.first %}
            {{- '<|im_start|>system\\n' + content + '<|im_end|>\\n' }}
        {%- elif message.role == "assistant" %}
            {%- set reasoning_content = '' %}
            {%- if message.reasoning_content is string %}
                {%- set reasoning_content = message.reasoning_content %}
            {%- elif '</think>' in content %}
                {%- set reasoning_content = content.split('</think>')[0].rstrip('\\n').split('<think>')[-1].lstrip('\\n') %}
                {%- set content = content.split('</think>')[-1].lstrip('\\n') %}
            {%- endif %}
            {%- if reasoning_content %}
                {{- '<|im_start|>assistant\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') + '<|im_end|>' + eos + '\\n' }}
            {%- else %}
                {{- '<|im_start|>assistant\\n' + content + '<|im_end|>' + eos + '\\n' }}
            {%- endif %}
        {%- endif %}
    {%- endfor %}
    {%- if add_generation_prompt %}
        {{- '<|im_start|>assistant\\n' }}
    {%- endif %}
    """

tokenizer.chat_template = get_chat_template()


## Locate the repository and load the 5×5 curriculum data


In [ ]:
import ast
import json
import random
import re
import sys

import pandas as pd
from datasets import Dataset

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "src" / "minesweeper").is_dir() and (candidate / "dataset").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository root or its notebook directory. "
        "Expected both src/minesweeper and dataset folders."
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATASET_PATH = PROJECT_ROOT / "dataset" / "live_teacher_thinking_new.jsonl"
CURRICULUM_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "curriculum"
FIVE_BY_FIVE_OUTPUT_ROOT = CURRICULUM_OUTPUT_ROOT / "5x5"
SIX_BY_SIX_OUTPUT_ROOT = CURRICULUM_OUTPUT_ROOT / "6x6"
FIVE_BY_FIVE_SFT_ADAPTER_DIR = FIVE_BY_FIVE_OUTPUT_ROOT / "sft_adapter"
FIVE_BY_FIVE_GRPO_ADAPTER_DIR = FIVE_BY_FIVE_OUTPUT_ROOT / "grpo_adapter"
SIX_BY_SIX_GRPO_ADAPTER_DIR = SIX_BY_SIX_OUTPUT_ROOT / "grpo_adapter"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)


In [ ]:
def action_validation_error(row):
    try:
        board = ast.literal_eval(row["input"]) if isinstance(row["input"], str) else row["input"]
        move = json.loads(row["output"]) if isinstance(row["output"], str) else row["output"]
        action = move.get("action")
        x, y = move.get("x"), move.get("y")
        if action not in {"reveal", "flag"}:
            return "invalid_action"
        if not isinstance(x, int) or not isinstance(y, int):
            return "non_integer_coordinates"
        if not (0 <= y < len(board) and 0 <= x < len(board[y])):
            return "out_of_bounds"
        if str(board[y][x]) not in {".", "_"}:
            return "target_not_hidden"
        return ""
    except Exception as exc:
        return f"parse_error:{type(exc).__name__}"

all_rows = pd.read_json(DATASET_PATH, lines=True)
clean_rows = all_rows[
    (all_rows["max_rows"] == 5)
    & (all_rows["max_columns"] == 5)
    & (all_rows["stage_name"] == "thinking-sft")
].copy()

clean_rows["quality_error"] = clean_rows.apply(action_validation_error, axis=1)
invalid_rows = clean_rows[clean_rows["quality_error"].ne("")]

if not invalid_rows.empty:
    display(invalid_rows[["session_id", "game_id", "turn_index", "quality_error"]])
    raise ValueError(f"Found {len(invalid_rows)} invalid 5x5 training rows.")

assert clean_rows["thinking_text"].fillna("").str.strip().ne("").all()
print(f"All dataset rows: {len(all_rows)}")
print(f"Clean 5x5 rows loaded: {len(clean_rows)}")

if len(all_rows) == 1442:
    assert len(clean_rows) == 249, "Expected 249 clean 5x5 rows in the current dataset."


## Grouped split and token-safe SFT formatting


In [ ]:
def canonical_output(value):
    payload = json.loads(value) if isinstance(value, str) else value
    return json.dumps(
        {"action": payload["action"], "x": int(payload["x"]), "y": int(payload["y"])},
        separators=(",", ":"),
    )

def build_messages(row, thinking_text):
    assistant = f"<think>\n{thinking_text.strip()}\n</think>\n\n{canonical_output(row['output'])}"
    return [
        {"role": "system", "content": THINKING_SYSTEM_PROMPT},
        {"role": "user", "content": str(row["user_prompt"]).strip()},
        {"role": "assistant", "content": assistant},
    ]

def render_messages(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def format_sft_row(row):
    thinking = str(row["thinking_text"]).strip()
    rendered = render_messages(build_messages(row, thinking))
    original_tokens = token_count(rendered)
    was_trimmed = False

    if original_tokens > MAX_SEQ_LENGTH:
        empty_reasoning_text = render_messages(build_messages(row, ""))
        reserved_tokens = token_count(empty_reasoning_text) + 24
        available = max(64, MAX_SEQ_LENGTH - reserved_tokens)
        reasoning_ids = tokenizer(thinking, add_special_tokens=False)["input_ids"]
        keep = min(len(reasoning_ids), available)

        while keep >= 64:
            head_count = max(1, int(keep * 0.60))
            tail_count = max(1, keep - head_count)
            shortened = (
                tokenizer.decode(reasoning_ids[:head_count], skip_special_tokens=True).strip()
                + "\n...\n"
                + tokenizer.decode(reasoning_ids[-tail_count:], skip_special_tokens=True).strip()
            )
            rendered = render_messages(build_messages(row, shortened))
            if token_count(rendered) <= MAX_SEQ_LENGTH:
                was_trimmed = True
                break
            keep -= 64
        else:
            raise ValueError(f"Could not fit row {row.name} into {MAX_SEQ_LENGTH} tokens.")

    final_tokens = token_count(rendered)
    assert final_tokens <= MAX_SEQ_LENGTH
    assert "</think>" in rendered
    assert canonical_output(row["output"]) in rendered
    return pd.Series(
        {"text": rendered, "token_count": final_tokens, "was_trimmed": was_trimmed}
    )

game_ids = sorted(clean_rows["game_id"].unique().tolist())
random.Random(SEED).shuffle(game_ids)
eval_game_count = max(1, round(len(game_ids) * 0.20))
eval_game_ids = set(game_ids[:eval_game_count])

formatted = clean_rows.apply(format_sft_row, axis=1)
prepared = pd.concat([clean_rows.reset_index(drop=True), formatted.reset_index(drop=True)], axis=1)

train_frame = prepared[~prepared["game_id"].isin(eval_game_ids)].copy()
eval_frame = prepared[prepared["game_id"].isin(eval_game_ids)].copy()

assert set(train_frame["game_id"]).isdisjoint(set(eval_frame["game_id"]))
assert train_frame["text"].str.contains("</think>", regex=False).all()
assert prepared["token_count"].max() <= MAX_SEQ_LENGTH

train_dataset = Dataset.from_pandas(
    train_frame[["text", "game_id", "turn_index", "token_count", "was_trimmed"]],
    preserve_index=False,
)
eval_dataset = Dataset.from_pandas(
    eval_frame[["text", "game_id", "turn_index", "token_count", "was_trimmed"]],
    preserve_index=False,
)

print(f"Train rows/games: {len(train_dataset)}/{train_frame['game_id'].nunique()}")
print(f"Eval rows/games: {len(eval_dataset)}/{eval_frame['game_id'].nunique()}")
print(f"Token-trimmed rows: {int(prepared['was_trimmed'].sum())}")
print(f"Maximum formatted length: {int(prepared['token_count'].max())} tokens")


## Stage 1 — 5×5 SFT for two epochs


In [ ]:
from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(
    output_dir=str(FIVE_BY_FIVE_OUTPUT_ROOT / "sft_checkpoints"),
    dataset_text_field="text",
    dataset_num_proc=1,
    dataloader_num_workers=0,
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=SFT_EPOCHS,
    learning_rate=2e-5,
    warmup_ratio=0.10,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    seed=SEED,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to=REPORT_TO,
    run_name="minesweeper-5x5-thinking-sft",
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_args,
)

print(f"SFT is configured for {SFT_EPOCHS} epochs.")


In [ ]:
FIVE_BY_FIVE_SFT_COMPLETE = False

if RUN_SFT:
    sft_train_result = sft_trainer.train()
    FIVE_BY_FIVE_SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FIVE_BY_FIVE_SFT_ADAPTER_DIR)
    tokenizer.save_pretrained(FIVE_BY_FIVE_SFT_ADAPTER_DIR)
    assert (FIVE_BY_FIVE_SFT_ADAPTER_DIR / "adapter_config.json").exists()
    FIVE_BY_FIVE_SFT_COMPLETE = True
    print("Saved 5x5 SFT LoRA adapter:", FIVE_BY_FIVE_SFT_ADAPTER_DIR)
else:
    print("SFT skipped. GRPO will require an existing saved SFT adapter.")


## Stage boundary — saved 5×5 SFT adapter → 5×5 GRPO

5×5 GRPO is gated on the saved SFT adapter. By default it continues with the same loaded model; set RELOAD_SAVED_SFT_FOR_5X5_GRPO=True to reload the saved 4-bit adapter in a fresh phase.


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from minesweeper.training_pipeline import (
    THINKING_GRPO_STAGE,
    build_live_dataset,
    build_reward_function,
    evaluate_policy,
    extract_last_json_object,
)

if "sft_trainer" in globals():
    del sft_trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_5X5_GRPO and RELOAD_SAVED_SFT_FOR_5X5_GRPO:
    if not (FIVE_BY_FIVE_SFT_ADAPTER_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"No saved 5x5 SFT adapter at {FIVE_BY_FIVE_SFT_ADAPTER_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(FIVE_BY_FIVE_SFT_ADAPTER_DIR),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        fast_inference=False,
        max_lora_rank=LORA_RANK,
        gpu_memory_utilization=0.75,
    )
    tokenizer.chat_template = get_chat_template()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Reloaded saved 5x5 SFT adapter for 5x5 GRPO.")
elif RUN_5X5_GRPO:
    if not FIVE_BY_FIVE_SFT_COMPLETE or not (FIVE_BY_FIVE_SFT_ADAPTER_DIR / "adapter_config.json").exists():
        raise RuntimeError("5x5 GRPO cannot start until 5x5 SFT completes and its adapter is saved.")
    print("Using the saved 5x5 SFT state already loaded in memory.")

model.generation_config.max_length = None

five_by_five_live_train_dataset = build_live_dataset(
    tokenizer,
    system_prompt=THINKING_SYSTEM_PROMPT,
    stage=THINKING_GRPO_STAGE,
    num_examples=FIVE_BY_FIVE_GRPO_NUM_EXAMPLES,
    board_sizes=[(5, 5)],
    mine_densities=[0.15, 0.30],
    seed=SEED,
)

print(five_by_five_live_train_dataset)
print(five_by_five_live_train_dataset[0])


## Solver, format, and optional LLM-judge rewards


In [ ]:
solver_reward = build_reward_function(THINKING_GRPO_STAGE)
THINK_OUTPUT_RE = re.compile(
    r"\s*<think>\s*(?P<thinking>.+?)\s*</think>\s*(?P<payload>\{.*\})\s*\Z",
    re.DOTALL,
)

def completion_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        return str(completion.get("content", ""))
    if isinstance(completion, list) and completion:
        return completion_text(completion[-1])
    return str(completion or "")

def thinking_format_reward(prompts=None, completions=None, **kwargs):
    rewards = []
    logs = []
    for completion in completions or []:
        text = completion_text(completion)
        match = THINK_OUTPUT_RE.fullmatch(text)
        score = -0.35
        reason = "missing_or_extra_structure"

        if match:
            thinking = match.group("thinking").strip()
            try:
                payload = json.loads(match.group("payload"))
                valid_keys = {"action", "x", "y"}.issubset(payload)
                if thinking and valid_keys:
                    score = 0.20
                    reason = "valid_think_then_json"
                else:
                    score = -0.50
                    reason = "empty_thinking_or_missing_keys"
            except Exception:
                score = -0.50
                reason = "malformed_terminal_json"

        rewards.append(float(score))
        logs.append({"reward": float(score), "component": reason})

    thinking_format_reward.last_logs = logs
    return rewards

JUDGE_ENABLED = False
JUDGE_BASE_URL = os.getenv("JUDGE_BASE_URL")
JUDGE_API_KEY = os.getenv("JUDGE_API_KEY")
JUDGE_MODEL = os.getenv("JUDGE_MODEL")
JUDGE_WEIGHT = 0.25

judge_client = None
if JUDGE_ENABLED:
    if not all([JUDGE_BASE_URL, JUDGE_API_KEY, JUDGE_MODEL]):
        raise ValueError("Set JUDGE_BASE_URL, JUDGE_API_KEY, and JUDGE_MODEL.")
    from openai import OpenAI
    judge_client = OpenAI(base_url=JUDGE_BASE_URL, api_key=JUDGE_API_KEY)

def llm_reasoning_judge(prompts=None, completions=None, **kwargs):
    board_states = kwargs.get("board_state") or []
    rows = kwargs.get("rows") or []
    columns = kwargs.get("columns") or []
    rewards = []
    logs = []

    for index, completion in enumerate(completions or []):
        if not JUDGE_ENABLED:
            rewards.append(0.0)
            logs.append({"reward": 0.0, "component": "judge_disabled"})
            continue

        visible_board = board_states[index] if index < len(board_states) else "[missing]"
        text = completion_text(completion)
        judge_prompt = f"""Grade whether this Minesweeper reasoning is grounded only in the visible board and supports its final move.

Board dimensions: {rows[index]} rows x {columns[index]} columns
Visible board: {visible_board}
Completion:
{text}

Return JSON only:
{{"score": number from -1 to 1, "verdict": "correct|uncertain|incorrect", "issue": "short explanation"}}
Do not assume hidden mine locations and do not reward verbosity."""

        try:
            response = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": "You are a strict Minesweeper process-reasoning verifier."},
                    {"role": "user", "content": judge_prompt},
                ],
                temperature=0,
                max_tokens=5000,
            )
            judge_text = response.choices[0].message.content or ""
            payload = json.loads(extract_last_json_object(judge_text))
            score = max(-1.0, min(1.0, float(payload["score"])))
            logs.append(
                {
                    "reward": score,
                    "component": "llm_judge",
                    "verdict": payload.get("verdict"),
                    "issue": payload.get("issue"),
                }
            )
        except Exception as exc:
            score = 0.0
            logs.append(
                {
                    "reward": 0.0,
                    "component": "llm_judge_failure",
                    "issue": f"{type(exc).__name__}: {exc}",
                }
            )
        rewards.append(float(score))

    llm_reasoning_judge.last_logs = logs
    return rewards

reward_functions = [solver_reward, thinking_format_reward]
reward_weights = [1.0, 1.0]
if JUDGE_ENABLED:
    reward_functions.append(llm_reasoning_judge)
    reward_weights.append(JUDGE_WEIGHT)

print("Reward functions:", [getattr(fn, "__name__", type(fn).__name__) for fn in reward_functions])
print("Reward weights:", reward_weights)


In [ ]:
sample = five_by_five_live_train_dataset[0]
sample_board = ast.literal_eval(sample["board_state"])
sample_hidden = next(
    (x, y)
    for y, row in enumerate(sample_board)
    for x, value in enumerate(row)
    if str(value) in {".", "_"}
)
sample_completion = (
    "<think>\nInspect the visible constraints and choose a currently hidden cell.\n</think>\n\n"
    + json.dumps(
        {"action": "reveal", "x": sample_hidden[0], "y": sample_hidden[1]},
        separators=(",", ":"),
    )
)

solver_smoke = solver_reward(
    completions=[sample_completion],
    rows=[sample["rows"]],
    columns=[sample["columns"]],
    snapshot=[sample["snapshot"]],
)
format_smoke = thinking_format_reward(completions=[sample_completion])
missing_think_smoke = thinking_format_reward(
    completions=[json.dumps({"action": "reveal", "x": sample_hidden[0], "y": sample_hidden[1]})]
)
malformed_smoke = solver_reward(
    completions=["not-json"],
    rows=[sample["rows"]],
    columns=[sample["columns"]],
    snapshot=[sample["snapshot"]],
)

assert len(solver_smoke) == 1 and isinstance(solver_smoke[0], float)
assert format_smoke == [0.20]
assert missing_think_smoke == [-0.35]
assert malformed_smoke == [-1.0]
print(
    {
        "solver_reward": solver_smoke,
        "format_reward": format_smoke,
        "missing_think_reward": missing_think_smoke,
        "malformed_reward": malformed_smoke,
    }
)


## Configure and run 5×5 GRPO


In [ ]:
five_by_five_grpo_args = GRPOConfig(
    output_dir=str(FIVE_BY_FIVE_OUTPUT_ROOT / "grpo_checkpoints"),
    use_vllm=True,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.10,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_generations=8,
    temperature=1.0,
    top_p=0.95,
    top_k=50,
    max_prompt_length=MAX_SEQ_LENGTH - GRPO_MAX_COMPLETION_LENGTH,
    max_completion_length=GRPO_MAX_COMPLETION_LENGTH,
    max_steps=FIVE_BY_FIVE_GRPO_MAX_STEPS,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    reward_weights=reward_weights,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to=REPORT_TO,
    run_name="minesweeper-5x5-thinking-grpo",
    seed=SEED,
)

five_by_five_grpo_trainer = None
if RUN_5X5_GRPO:
    five_by_five_grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_functions,
        args=five_by_five_grpo_args,
        train_dataset=five_by_five_live_train_dataset,
    )
    print(f"5x5 GRPO is configured for {FIVE_BY_FIVE_GRPO_MAX_STEPS} steps.")
else:
    print("5x5 GRPO trainer construction skipped.")


In [ ]:
evaluation_history = {}

def evaluate_current_policy(label, board_sizes):
    FastLanguageModel.for_inference(model)
    try:
        records, summary = evaluate_policy(
            model=model,
            tokenizer=tokenizer,
            system_prompt=THINKING_SYSTEM_PROMPT,
            stage=THINKING_GRPO_STAGE,
            board_sizes=board_sizes,
            mine_densities=[0.15, 0.30],
            games_per_setting=2,
            seed=SEED,
            max_turns=50,
        )
        evaluation_history[label] = summary
        print(label, summary)
        return records, summary
    finally:
        if hasattr(FastLanguageModel, "for_training"):
            FastLanguageModel.for_training(model)

if RUN_STAGE_EVALUATION:
    evaluate_current_policy("after_5x5_sft_on_5x5", [(5, 5)])


In [ ]:
FIVE_BY_FIVE_GRPO_COMPLETE = False

if RUN_5X5_GRPO:
    assert (FIVE_BY_FIVE_SFT_ADAPTER_DIR / "adapter_config.json").exists(), "Saved 5x5 SFT adapter is required."
    five_by_five_grpo_train_result = five_by_five_grpo_trainer.train()
    FIVE_BY_FIVE_GRPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FIVE_BY_FIVE_GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(FIVE_BY_FIVE_GRPO_ADAPTER_DIR)
    assert (FIVE_BY_FIVE_GRPO_ADAPTER_DIR / "adapter_config.json").exists()
    FIVE_BY_FIVE_GRPO_COMPLETE = True
    print("Saved 5x5 GRPO LoRA adapter:", FIVE_BY_FIVE_GRPO_ADAPTER_DIR)

    if RUN_STAGE_EVALUATION:
        evaluate_current_policy("after_5x5_grpo_on_5x5", [(5, 5)])
        display(pd.DataFrame(evaluation_history).T)
else:
    print("5x5 GRPO skipped.")


## Stage boundary — saved 5×5 GRPO adapter → 6×6 GRPO

The 6×6 stage uses live states only and performs no additional SFT. By default it continues from the trained model in memory; set RELOAD_SAVED_5X5_GRPO_FOR_6X6=True to resume from the saved 5×5 GRPO adapter.


In [ ]:
six_by_six_live_train_dataset = None

if RUN_6X6_GRPO:
    five_by_five_grpo_artifact = FIVE_BY_FIVE_GRPO_ADAPTER_DIR / "adapter_config.json"
    if RELOAD_SAVED_5X5_GRPO_FOR_6X6:
        if not five_by_five_grpo_artifact.exists():
            raise FileNotFoundError(
                f"No saved 5x5 GRPO adapter at {FIVE_BY_FIVE_GRPO_ADAPTER_DIR}"
            )
    elif not FIVE_BY_FIVE_GRPO_COMPLETE or not five_by_five_grpo_artifact.exists():
        raise RuntimeError(
            "6x6 GRPO requires a completed 5x5 GRPO stage. "
            "To resume from an existing artifact, set "
            "RELOAD_SAVED_5X5_GRPO_FOR_6X6=True."
        )

    if "five_by_five_grpo_trainer" in globals():
        del five_by_five_grpo_trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if RELOAD_SAVED_5X5_GRPO_FOR_6X6:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(FIVE_BY_FIVE_GRPO_ADAPTER_DIR),
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
            fast_inference=False,
            max_lora_rank=LORA_RANK,
            gpu_memory_utilization=0.75,
        )
        tokenizer.chat_template = get_chat_template()
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print("Reloaded the saved 5x5 GRPO adapter for 6x6 GRPO.")
    else:
        print("Continuing from the completed 5x5 GRPO model in memory.")

    model.generation_config.max_length = None
    six_by_six_live_train_dataset = build_live_dataset(
        tokenizer,
        system_prompt=THINKING_SYSTEM_PROMPT,
        stage=THINKING_GRPO_STAGE,
        num_examples=SIX_BY_SIX_GRPO_NUM_EXAMPLES,
        board_sizes=[(6, 6)],
        mine_densities=[0.15, 0.30],
        seed=SEED + 1,
    )
    assert set(six_by_six_live_train_dataset["rows"]) == {6}
    assert set(six_by_six_live_train_dataset["columns"]) == {6}
    print(six_by_six_live_train_dataset)
    print(six_by_six_live_train_dataset[0])
else:
    print("6x6 GRPO skipped.")


In [ ]:
if RUN_6X6_GRPO:
    six_by_six_sample = six_by_six_live_train_dataset[0]
    six_by_six_board = ast.literal_eval(six_by_six_sample["board_state"])
    six_by_six_hidden = next(
        (x, y)
        for y, row in enumerate(six_by_six_board)
        for x, value in enumerate(row)
        if str(value) in {".", "_"}
    )
    six_by_six_sample_completion = (
        "<think>\nInspect the visible constraints and choose a currently hidden cell.\n</think>\n\n"
        + json.dumps(
            {"action": "reveal", "x": six_by_six_hidden[0], "y": six_by_six_hidden[1]},
            separators=(",", ":"),
        )
    )
    six_by_six_solver_smoke = solver_reward(
        completions=[six_by_six_sample_completion],
        rows=[six_by_six_sample["rows"]],
        columns=[six_by_six_sample["columns"]],
        snapshot=[six_by_six_sample["snapshot"]],
    )
    six_by_six_format_smoke = thinking_format_reward(
        completions=[six_by_six_sample_completion]
    )
    assert len(six_by_six_solver_smoke) == 1
    assert isinstance(six_by_six_solver_smoke[0], float)
    assert six_by_six_format_smoke == [0.20]
    print(
        {
            "6x6_solver_reward": six_by_six_solver_smoke,
            "6x6_format_reward": six_by_six_format_smoke,
        }
    )


## Configure and run 6×6 GRPO


In [ ]:
six_by_six_grpo_trainer = None

if RUN_6X6_GRPO:
    six_by_six_grpo_args = GRPOConfig(
        output_dir=str(SIX_BY_SIX_OUTPUT_ROOT / "grpo_checkpoints"),
        use_vllm=True,
        learning_rate=5e-6,
        weight_decay=0.001,
        warmup_ratio=0.10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_generations=8,
        temperature=1.0,
        top_p=0.95,
        top_k=50,
        max_prompt_length=MAX_SEQ_LENGTH - GRPO_MAX_COMPLETION_LENGTH,
        max_completion_length=GRPO_MAX_COMPLETION_LENGTH,
        max_steps=SIX_BY_SIX_GRPO_MAX_STEPS,
        save_steps=50,
        save_total_limit=3,
        max_grad_norm=1.0,
        remove_unused_columns=False,
        reward_weights=reward_weights,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        report_to=REPORT_TO,
        run_name="minesweeper-6x6-thinking-grpo",
        seed=SEED + 1,
    )

    six_by_six_grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_functions,
        args=six_by_six_grpo_args,
        train_dataset=six_by_six_live_train_dataset,
    )
    print(f"6x6 GRPO is configured for {SIX_BY_SIX_GRPO_MAX_STEPS} steps.")


In [ ]:
SIX_BY_SIX_GRPO_COMPLETE = False

if RUN_6X6_GRPO:
    assert (FIVE_BY_FIVE_GRPO_ADAPTER_DIR / "adapter_config.json").exists(), (
        "Saved 5x5 GRPO adapter is required."
    )
    six_by_six_grpo_train_result = six_by_six_grpo_trainer.train()
    SIX_BY_SIX_GRPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(SIX_BY_SIX_GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(SIX_BY_SIX_GRPO_ADAPTER_DIR)
    assert (SIX_BY_SIX_GRPO_ADAPTER_DIR / "adapter_config.json").exists()
    SIX_BY_SIX_GRPO_COMPLETE = True
    print("Saved 6x6 GRPO LoRA adapter:", SIX_BY_SIX_GRPO_ADAPTER_DIR)

    if RUN_STAGE_EVALUATION:
        evaluate_current_policy("after_6x6_grpo_on_5x5", [(5, 5)])
        evaluate_current_policy("after_6x6_grpo_on_6x6", [(6, 6)])
        display(pd.DataFrame(evaluation_history).T)
else:
    print("6x6 GRPO skipped.")


## Training artifacts


In [ ]:
print("5x5 SFT adapter:", FIVE_BY_FIVE_SFT_ADAPTER_DIR)
print("5x5 GRPO adapter:", FIVE_BY_FIVE_GRPO_ADAPTER_DIR)
print("6x6 GRPO adapter:", SIX_BY_SIX_GRPO_ADAPTER_DIR)
print("5x5 SFT complete:", FIVE_BY_FIVE_SFT_COMPLETE)
print("5x5 GRPO complete:", FIVE_BY_FIVE_GRPO_COMPLETE)
print("6x6 GRPO complete:", SIX_BY_SIX_GRPO_COMPLETE)
